# Baseline ImageCAS1000 · fold 0 · 100 épocas (Dice + CE, sin clDice)

Se sigue el patrón del notebook 28 (v1@1000: copia local, entrenamiento y evaluación), adaptado al baseline con dos correcciones:

1. **Split forzado.** Se reutiliza `splits_final_Dataset503_1000_auto5fold.json`, el mismo de v1@1000, en lugar de permitir que nnU-Net genere uno nuevo. Es imprescindible para que baseline y v1 se comparen sobre los mismos 200 pacientes de validación (comparación pareada, Δ componentes, H1).
2. **Copia robusta con reintentos.** Sustituye al `shutil.copytree` simple (que en el notebook 28 requirió 269.9 min y no se recupera de un corte de FUSE) por una copia que omite los archivos ya copiados y remonta Drive en caso de fallo, evitando reiniciar desde cero.


## 0. Comprobación de la GPU


In [1]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Activa una GPU (A100 recomendado) antes de continuar.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")


PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.09 GB


## 1. Instalación de dependencias

Igual que en el notebook 28: si `pip install` actualiza paquetes ya cargados (numpy, torch, batchgenerators), el kernel se reinicia deliberadamente en la celda siguiente para evitar un runtime con módulos C desincronizados.


In [2]:
!python -m pip install -q nnunetv2 medpy scikit-image SimpleITK
print("Dependencias instaladas.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 14.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 126.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 1b. Reinicio del runtime tras la instalación

La ejecución reinicia el kernel, comportamiento previsto: continúa desde la celda de montaje de Drive.


In [ ]:
import os
os.kill(os.getpid(), 9)


## 2. Montaje de Google Drive


In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


Mounted at /content/drive


## 3. Definición de rutas y configuración


In [2]:
import os, shutil, json, time
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")

DRIVE_PREP_ROOT = BASE / "nnUNet_preprocessed"
DRIVE_DATASET = DRIVE_PREP_ROOT / "Dataset503_ImageCAS1000"

LOCAL_PREP_ROOT = Path("/content/nnUNet_preprocessed")
LOCAL_DATASET = LOCAL_PREP_ROOT / "Dataset503_ImageCAS1000"

NNUNET_RESULTS = BASE / "nnUNet_results"

DATASET_ID = 503
DATASET_FOLDER_NAME = "Dataset503_ImageCAS1000"
CONFIGURATION = "3d_fullres"
FOLD = 0
TRAINER_NAME = "nnUNetTrainer_100epochs"   # baseline: built-in, SIN clDice

# Split canónico ya utilizado por v1@1000 (notebook 28, sección 13); es obligatorio
# para que baseline y v1 sean comparables de forma pareada
SPLIT_CORRECTO = BASE / "splits_final_Dataset503_1000_auto5fold.json"
assert SPLIT_CORRECTO.exists(), "Falta splits_final_Dataset503_1000_auto5fold.json en BASE."

print("Drive dataset:", DRIVE_DATASET)
print("Local dataset:", LOCAL_DATASET)
print("Resultados:", NNUNET_RESULTS)
print("Split canonico:", SPLIT_CORRECTO)


Drive dataset: /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed/Dataset503_ImageCAS1000
Local dataset: /content/nnUNet_preprocessed/Dataset503_ImageCAS1000
Resultados: /content/drive/MyDrive/TFM_nnUNet/nnUNet_results
Split canonico: /content/drive/MyDrive/TFM_nnUNet/splits_final_Dataset503_1000_auto5fold.json


## 4. Verificación de que Dataset503 en Drive está completo (1000/1000)


In [3]:
drive_cfg = DRIVE_DATASET / "nnUNetPlans_3d_fullres"
drive_gt = DRIVE_DATASET / "gt_segmentations"

imgs = [p for p in drive_cfg.glob("ImageCAS_*.b2nd") if not p.name.endswith("_seg.b2nd")]
segs = list(drive_cfg.glob("ImageCAS_*_seg.b2nd"))
pkls = list(drive_cfg.glob("ImageCAS_*.pkl"))
gts = list(drive_gt.glob("ImageCAS_*.nii.gz"))

print("Imagenes:", len(imgs), "Segmentaciones:", len(segs), "PKL:", len(pkls), "GT:", len(gts))
for required in ["nnUNetPlans.json", "dataset.json", "dataset_fingerprint.json"]:
    print(required, "->", (DRIVE_DATASET / required).exists())

assert len(imgs) == 1000 and len(segs) == 1000 and len(pkls) == 1000 and len(gts) == 1000
assert all((DRIVE_DATASET / x).exists() for x in ["nnUNetPlans.json", "dataset.json", "dataset_fingerprint.json"])
print("\n✅ Dataset503 en Drive: 1000/1000 y archivos de planificacion presentes.")


Imagenes: 1000 Segmentaciones: 1000 PKL: 1000 GT: 1000
nnUNetPlans.json -> True
dataset.json -> True
dataset_fingerprint.json -> True

✅ Dataset503 en Drive: 1000/1000 y archivos de planificacion presentes.


## 5. Comprobación del espacio local


In [4]:
usage = shutil.disk_usage("/content")
free_gb = usage.free / 1024**3
print(f"Espacio libre local: {free_gb:.1f} GB")
if free_gb < 120:
    raise RuntimeError(f"Solo hay {free_gb:.1f} GB libres. No copies todavia.")
print("✅ Espacio local suficiente.")


Espacio libre local: 187.8 GB
✅ Espacio local suficiente.


## 6. Copia robusta y reanudable de Drive al disco local

A diferencia del `shutil.copytree` simple del notebook 28, si un archivo ya existe localmente con el mismo tamaño se omite (no se vuelve a copiar) y, ante un error de E/S, se reintenta remontando Drive en lugar de perder toda la copia.


In [5]:
def copiar_robusto(src: Path, dst: Path, max_reintentos=5):
    dst.mkdir(parents=True, exist_ok=True)
    archivos = [f for f in src.rglob("*") if f.is_file()]
    print(f"Total de archivos a copiar: {len(archivos)}")

    pendientes = archivos
    intento = 0
    while pendientes and intento < max_reintentos:
        intento += 1
        print(f"\n--- Intento {intento}/{max_reintentos}: {len(pendientes)} archivos pendientes ---")
        fallidos = []
        for i, f in enumerate(pendientes):
            rel = f.relative_to(src)
            destino = dst / rel
            try:
                if destino.exists() and destino.stat().st_size == f.stat().st_size:
                    continue
                destino.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, destino)
            except OSError as e:
                fallidos.append(f)
                if len(fallidos) % 50 == 1:
                    print(f"  fallo en {rel}: {e}")
            if i % 200 == 0:
                print(f"  {i}/{len(pendientes)} procesados ({len(fallidos)} fallos hasta ahora)")

        pendientes = fallidos
        if pendientes:
            print(f"Quedan {len(pendientes)} archivos fallidos. Remontando Drive antes de reintentar...")
            try:
                drive.flush_and_unmount()
            except Exception:
                pass
            time.sleep(10)
            drive.mount("/content/drive", force_remount=True)
            time.sleep(5)

    if pendientes:
        print(f"⚠️ Tras {max_reintentos} intentos siguen fallando {len(pendientes)} archivos.")
    else:
        print("✅ Copia completa (todos los archivos verificados por tamano).")
    return pendientes


if LOCAL_DATASET.exists():
    print("Ya existe una copia local parcial/previa -- se reutiliza y se completa (no se borra).")
LOCAL_DATASET.mkdir(parents=True, exist_ok=True)

for name in ["nnUNetPlans.json", "dataset.json", "dataset_fingerprint.json"]:
    shutil.copy2(DRIVE_DATASET / name, LOCAL_DATASET / name)
    print("Copiado:", name)

for folder_name in ["nnUNetPlans_3d_fullres", "gt_segmentations"]:
    print(f"\n=== Copiando {folder_name} ===")
    t0 = time.time()
    pendientes = copiar_robusto(DRIVE_DATASET / folder_name, LOCAL_DATASET / folder_name)
    print(f"{folder_name}: {(time.time()-t0)/60:.1f} min, {len(pendientes)} archivos sin copiar")
    assert not pendientes, f"Quedaron archivos sin copiar en {folder_name}, revisa antes de seguir."

# Split canónico: es la diferencia deliberada respecto al notebook 28
shutil.copy2(SPLIT_CORRECTO, LOCAL_DATASET / "splits_final.json")
print("\n✅ Copia local terminada, incluido el split canonico de 1000 casos.")


Copiado: nnUNetPlans.json
Copiado: dataset.json
Copiado: dataset_fingerprint.json

=== Copiando nnUNetPlans_3d_fullres ===
Total de archivos a copiar: 3000

--- Intento 1/5: 3000 archivos pendientes ---
  0/3000 procesados (0 fallos hasta ahora)
  200/3000 procesados (0 fallos hasta ahora)
  400/3000 procesados (0 fallos hasta ahora)
  600/3000 procesados (0 fallos hasta ahora)
  800/3000 procesados (0 fallos hasta ahora)
  1000/3000 procesados (0 fallos hasta ahora)
  1200/3000 procesados (0 fallos hasta ahora)
  1400/3000 procesados (0 fallos hasta ahora)
  1600/3000 procesados (0 fallos hasta ahora)
  1800/3000 procesados (0 fallos hasta ahora)
  2000/3000 procesados (0 fallos hasta ahora)
  2200/3000 procesados (0 fallos hasta ahora)
  2400/3000 procesados (0 fallos hasta ahora)
  2600/3000 procesados (0 fallos hasta ahora)
  2800/3000 procesados (0 fallos hasta ahora)
✅ Copia completa (todos los archivos verificados por tamano).
nnUNetPlans_3d_fullres: 260.5 min, 0 archivos sin co

## 7. Verificación de la copia local (1000/1000) y de la coincidencia del split con el canónico


In [6]:
local_cfg = LOCAL_DATASET / "nnUNetPlans_3d_fullres"
local_gt = LOCAL_DATASET / "gt_segmentations"

imgs_local = [p for p in local_cfg.glob("ImageCAS_*.b2nd") if not p.name.endswith("_seg.b2nd")]
segs_local = list(local_cfg.glob("ImageCAS_*_seg.b2nd"))
pkls_local = list(local_cfg.glob("ImageCAS_*.pkl"))
gts_local = list(local_gt.glob("ImageCAS_*.nii.gz"))

print("Imagenes:", len(imgs_local), "Segmentaciones:", len(segs_local), "PKL:", len(pkls_local), "GT:", len(gts_local))
assert len(imgs_local) == 1000 and len(segs_local) == 1000 and len(pkls_local) == 1000 and len(gts_local) == 1000

with open(LOCAL_DATASET / "splits_final.json") as f:
    split_local = json.load(f)
with open(SPLIT_CORRECTO) as f:
    split_canonico = json.load(f)
assert split_local == split_canonico, "El split local NO coincide con el canonico -- no seria comparable con v1@1000."

fold0 = split_local[FOLD]
print(f"Fold {FOLD}: {len(fold0['train'])} train / {len(fold0['val'])} val")
assert len(fold0["train"]) == 800 and len(fold0["val"]) == 200

print("\n✅ Copia local integra: 1000/1000, split identico al de v1@1000.")


Imagenes: 1000 Segmentaciones: 1000 PKL: 1000 GT: 1000
Fold 0: 800 train / 200 val

✅ Copia local integra: 1000/1000, split identico al de v1@1000.


## 8. Configuración de nnU-Net: datos en el SSD local y resultados en Drive


In [7]:
os.environ["nnUNet_preprocessed"] = str(LOCAL_PREP_ROOT)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)
NNUNET_RAW = BASE / "nnUNet_raw"
os.environ["nnUNet_raw"] = str(NNUNET_RAW)

GT_FOLDER = DRIVE_DATASET / "gt_segmentations"  # persistente, para evaluar aunque /content ya no exista

print("nnUNet_preprocessed =", os.environ["nnUNet_preprocessed"])
print("nnUNet_results      =", os.environ["nnUNet_results"])


nnUNet_preprocessed = /content/nnUNet_preprocessed
nnUNet_results      = /content/drive/MyDrive/TFM_nnUNet/nnUNet_results


## 9. Funciones de métricas (idénticas a las del notebook 28 para garantizar la comparabilidad)


In [8]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage
from skimage.morphology import skeletonize
from medpy.metric.binary import hd95 as medpy_hd95

CONECTIVIDAD_26 = np.ones((3, 3, 3), dtype=int)

def cargar_volumen(path):
    img = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(img).astype(np.uint8)
    spacing = img.GetSpacing()[::-1]
    return array, spacing

def dice_score(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    return (2.0 * inter) / (pred.sum() + gt.sum() + 1e-8)

def hard_cldice(pred, gt):
    skel_pred = skeletonize(pred.astype(bool))
    skel_gt = skeletonize(gt.astype(bool))
    tprec = (skel_pred & gt).sum() / (skel_pred.sum() + 1e-8)
    tsens = (skel_gt & pred).sum() / (skel_gt.sum() + 1e-8)
    return (2 * tprec * tsens) / (tprec + tsens + 1e-8)

def componentes_conexas(pred):
    labeled, n = ndimage.label(pred, structure=CONECTIVIDAD_26)
    return labeled, n

def cobertura_componente_principal(pred, gt):
    labeled, n = componentes_conexas(pred)
    if n == 0:
        return 0.0, 0
    tamanos = ndimage.sum(pred, labeled, index=range(1, n + 1))
    idx_mayor = np.argmax(tamanos) + 1
    componente_mayor = labeled == idx_mayor
    cobertura = (componente_mayor & gt.astype(bool)).sum() / (gt.sum() + 1e-8)
    return cobertura, n

def hd95_seguro(pred, gt, spacing):
    if pred.sum() == 0 or gt.sum() == 0:
        return np.nan
    return medpy_hd95(pred, gt, voxelspacing=spacing)

def calcular_metricas_caso(pred_path, gt_path):
    pred, spacing = cargar_volumen(pred_path)
    gt, _ = cargar_volumen(gt_path)
    cobertura, n_componentes = cobertura_componente_principal(pred, gt)
    return {
        "dice": dice_score(pred, gt),
        "cldice": hard_cldice(pred, gt),
        "cobertura_principal": cobertura,
        "n_componentes": n_componentes,
        "hd95_crudo_mm": hd95_seguro(pred, gt, spacing),
    }

def evaluar_trainer(trainer_name):
    carpeta_val = (
        NNUNET_RESULTS / DATASET_FOLDER_NAME
        / f"{trainer_name}__nnUNetPlans__{CONFIGURATION}"
        / f"fold_{FOLD}" / "validation"
    )
    preds = sorted(carpeta_val.glob("*.nii.gz"))
    if not preds:
        raise RuntimeError(f"No se encontraron predicciones en {carpeta_val}")

    filas = []
    for pred_path in preds:
        case_id = pred_path.name.replace(".nii.gz", "")
        gt_path = GT_FOLDER / pred_path.name
        if not gt_path.exists():
            continue
        m = calcular_metricas_caso(pred_path, gt_path)
        m["case_id"] = case_id
        filas.append(m)

    df = pd.DataFrame(filas)
    resumen = {
        "dice_mean": df["dice"].mean(),
        "cldice_mean": df["cldice"].mean(),
        "n_componentes_mean": df["n_componentes"].mean(),
        "hd95_crudo_mean": df["hd95_crudo_mm"].mean(skipna=True),
        "n_casos": len(df),
    }
    return df, resumen

print("Funciones de metricas cargadas.")


Funciones de metricas cargadas.


## 10. Comprobación final previa al entrenamiento

No se requiere trainer personalizado: el baseline utiliza el trainer integrado `nnUNetTrainer_100epochs` (Dice + CE, sin clDice).


In [9]:
print("=== CHECK FINAL ===")
print("Dataset local:", LOCAL_DATASET)
print("Trainer:", TRAINER_NAME, "(built-in, sin modificar)")
print("Imagenes locales:", len(imgs_local), "| Segmentaciones locales:", len(segs_local))
print("Split local == canonico:", split_local == split_canonico)

# Aviso si ya existe un entrenamiento previo con este trainer, para evitar relanzarlo por error
posible_previo = NNUNET_RESULTS / DATASET_FOLDER_NAME / f"{TRAINER_NAME}__nnUNetPlans__{CONFIGURATION}" / f"fold_{FOLD}"
if posible_previo.exists():
    print(f"\n⚠️ Ya existe una carpeta de resultados en {posible_previo}. Debe comprobarse si ya se entrenó antes de lanzar de nuevo.")
else:
    print("\nNo hay entrenamiento previo de este trainer/fold -- listo para lanzar.")

assert len(imgs_local) == 1000 and len(segs_local) == 1000
assert split_local == split_canonico
print("\n✅ LISTO PARA ENTRENAR.")


=== CHECK FINAL ===
Dataset local: /content/nnUNet_preprocessed/Dataset503_ImageCAS1000
Trainer: nnUNetTrainer_100epochs (built-in, sin modificar)
Imagenes locales: 1000 | Segmentaciones locales: 1000
Split local == canonico: True

No hay entrenamiento previo de este trainer/fold -- listo para lanzar.

✅ LISTO PARA ENTRENAR.


## 11. Entrenamiento (100 épocas, baseline sin clDice)

Las celdas posteriores se ejecutan una vez finalizado el entrenamiento.


In [10]:
!nnUNetv2_train 503 3d_fullres 0 -tr nnUNetTrainer_100epochs --npz



############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-09-20 15:50:10.382527: Using torch.compile...
2026-09-20 15:50:13.591020: do_dummy_2d_data_aug: False
2026-09-20 15:50:13.596697: Using splits from existing split file: /content/nnUNet_preprocessed/Dataset503_ImageCAS1000/splits_final.json
2026-09-20 15:50:13.599839: The split file contains

## 11b. Desconexión de Colab durante el entrenamiento

Si el runtime sigue activo (solo se interrumpió la celda) y `/content` conserva la copia local intacta con su `splits_final.json`, el entrenamiento se reanuda con `--c`:


In [11]:
# Solo si el entrenamiento se interrumpió y /content conserva la copia local:
#!nnUNetv2_train 503 3d_fullres 0 -tr nnUNetTrainer_100epochs --npz --c

# Si se perdió el runtime completo (p. ej., por un cambio de tipo de hardware),
# se vuelve a ejecutar desde la sección 2: la copia robusta de la sección 6 omite
# los archivos que sigan intactos en /content si este sobrevivió,
# o copia todo de nuevo si el runtime es nuevo.


## 12. Evaluación y guardado de métricas caso a caso


In [12]:
df_metricas, resumen = evaluar_trainer(TRAINER_NAME)

print("=== Resultados: baseline (Dice+CE), Dataset503, fold 0, 100 epocas ===")
print(f"Dice:          {resumen['dice_mean']:.4f}")
print(f"clDice:        {resumen['cldice_mean']:.4f}")
print(f"n_componentes: {resumen['n_componentes_mean']:.2f}")
print(f"HD95 crudo:    {resumen['hd95_crudo_mean']:.2f} mm")
print(f"n_casos:       {resumen['n_casos']}")

csv_out = BASE / "baseline_1000casos_100epochs_metricas_por_caso.csv"
df_metricas.to_csv(csv_out, index=False)
print("\nGuardado:", csv_out)


=== Resultados: baseline (Dice+CE), Dataset503, fold 0, 100 epocas ===
Dice:          0.7516
clDice:        0.8136
n_componentes: 16.77
HD95 crudo:    23.44 mm
n_casos:       200

Guardado: /content/drive/MyDrive/TFM_nnUNet/baseline_1000casos_100epochs_metricas_por_caso.csv


## 13. Comparación pareada con v1@1000 (mismo split, mismos 200 pacientes)


In [13]:
from scipy import stats

df_v1 = pd.read_csv(BASE / "v1_1000casos_100epochs_metricas_por_caso.csv")
df_v1 = df_v1.sort_values("case_id").reset_index(drop=True)
df_base = df_metricas.sort_values("case_id").reset_index(drop=True)

assert list(df_v1["case_id"]) == list(df_base["case_id"]), \
    "case_id no coinciden entre baseline y v1 -- revisa el split antes de tratarlo como pareado."

delta_componentes = df_base["n_componentes"] - df_v1["n_componentes"]

resumen_tabla = pd.DataFrame([
    {"modelo": "baseline (Dice+CE)", "dice": resumen["dice_mean"], "cldice": resumen["cldice_mean"],
     "n_componentes": resumen["n_componentes_mean"], "hd95_mm_medpy": resumen["hd95_crudo_mean"]},
    {"modelo": "v1 λ=.2 (1000 casos)", "dice": df_v1["dice"].mean(), "cldice": df_v1["cldice"].mean(),
     "n_componentes": df_v1["n_componentes"].mean(), "hd95_mm_medpy": df_v1["hd95_crudo_mm"].mean()},
])
print(resumen_tabla.to_string(index=False))

print(f"\nΔcomponentes medio (baseline - v1) = {delta_componentes.mean():.3f}")
print("(positivo => v1 tiene MENOS componentes que baseline, i.e. v1 reduce fragmentacion, apoyando H1)")

print("\n=== Tests pareados (mismos 200 pacientes) ===")
for metrica_base, metrica_v1, nombre in [
    ("dice", "dice", "Dice"),
    ("cldice", "cldice", "clDice"),
    ("n_componentes", "n_componentes", "n_componentes"),
    ("hd95_crudo_mm", "hd95_crudo_mm", "HD95"),
]:
    a = df_base[metrica_base].values
    b = df_v1[metrica_v1].values
    mask = ~(np.isnan(a) | np.isnan(b))
    w_stat, w_p = stats.wilcoxon(a[mask], b[mask])
    t_stat, t_p = stats.ttest_rel(a[mask], b[mask])
    print(f"{nombre}: Wilcoxon p={w_p:.4g} | ttest_rel p={t_p:.4g} (n={mask.sum()})")

csv_delta = BASE / "delta_componentes_1000casos_baseline_vs_v1.csv"
pd.DataFrame({
    "case_id": df_base["case_id"],
    "n_componentes_baseline": df_base["n_componentes"],
    "n_componentes_v1": df_v1["n_componentes"],
    "delta_componentes": delta_componentes,
}).to_csv(csv_delta, index=False)
print("\nGuardado:", csv_delta)


              modelo     dice   cldice  n_componentes  hd95_mm_medpy
  baseline (Dice+CE) 0.751640 0.813646         16.770      23.444308
v1 λ=.2 (1000 casos) 0.747547 0.811284         16.985      24.719474

Δcomponentes medio (baseline - v1) = -0.215
(positivo => v1 tiene MENOS componentes que baseline, i.e. v1 reduce fragmentacion, apoyando H1)

=== Tests pareados (mismos 200 pacientes) ===
Dice: Wilcoxon p=0.000172 | ttest_rel p=0.0001431 (n=200)
clDice: Wilcoxon p=0.1287 | ttest_rel p=0.0809 (n=200)
n_componentes: Wilcoxon p=0.5716 | ttest_rel p=0.4177 (n=200)
HD95: Wilcoxon p=0.0005125 | ttest_rel p=0.002011 (n=200)

Guardado: /content/drive/MyDrive/TFM_nnUNet/delta_componentes_1000casos_baseline_vs_v1.csv


## Archivos persistentes

- Checkpoints y resultados → `MyDrive/TFM_nnUNet/nnUNet_results`
- Métricas del baseline → `baseline_1000casos_100epochs_metricas_por_caso.csv`
- Δ componentes y tests pareados → `delta_componentes_1000casos_baseline_vs_v1.csv`

La copia de `/content/nnUNet_preprocessed` es temporal y desaparece al cerrar el runtime; los datos originales de Drive no se modifican.
